#1,加载 train_feat.csv，拆分特征 X 和标签 y
#2,train_test_split + StratifiedKFold 交叉验证
#3,在多模型（LogisticRegression、RandomForest、XGBoost）上跑 baseline，记录各自 CV 分数
#4,选定 RandomForest（或最佳模型）进行 final training + hold-out 测试，输出 accuracy、classification_report
#5,将训练好的模型 joblib.dump(model, '../models/rf.joblib')
#6,把基本性能指标写入 results/metrics.txt

In [1]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier

In [2]:
# ── 1. 加载数据，拆分 X / y ───────────────────────────────
df = pd.read_csv('../data/processed/train_feat.csv')
print(f"数据维度: {df.shape}")
print(f"列名: {df.columns.tolist()}")

X = df.drop(columns=['Survived'])
y = df['Survived']
print(f"\nX: {X.shape}, y: {y.shape}")
print(f"y 分布:\n{y.value_counts()}")

数据维度: (891, 9)
列名: ['Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'Sex_male', 'Embarked_Q', 'Embarked_S']

X: (891, 8), y: (891,)
y 分布:
Survived
0    549
1    342
Name: count, dtype: int64


In [3]:
# ── 2. train_test_split ───────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y          # 保持正负样本比例
)
print(f"\ntrain: {X_train.shape}, test: {X_test.shape}")


train: (712, 8), test: (179, 8)


In [4]:
# ── 3. 多模型 CV baseline ─────────────────────────────────
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=42),
    'RandomForest':       RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost':            XGBClassifier(n_estimators=100, random_state=42,
                                        eval_metric='logloss', verbosity=0),
}

cv_results = {}
print("\n── CV 结果(5-Fold Accuracy)──")
for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train,
                             cv=skf, scoring='accuracy')
    cv_results[name] = scores
    print(f"{name:22s}  mean={scores.mean():.4f}  std={scores.std():.4f}")


── CV 结果(5-Fold Accuracy)──
LogisticRegression      mean=0.7935  std=0.0273
RandomForest            mean=0.8076  std=0.0212
XGBoost                 mean=0.8006  std=0.0239


In [5]:
# ── 4. 选定最佳模型，final training + hold-out 测试 ────────
best_name = max(cv_results, key=lambda k: cv_results[k].mean())
print(f"\n[INFO] 最佳模型: {best_name}")

best_model = models[best_name]
best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_test)
acc    = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred, target_names=['Not Survived', 'Survived'])

print(f"\nHold-out Accuracy: {acc:.4f}")
print("\nClassification Report:")
print(report)


[INFO] 最佳模型: RandomForest

Hold-out Accuracy: 0.8156

Classification Report:
              precision    recall  f1-score   support

Not Survived       0.83      0.87      0.85       110
    Survived       0.78      0.72      0.75        69

    accuracy                           0.82       179
   macro avg       0.81      0.80      0.80       179
weighted avg       0.81      0.82      0.81       179



In [6]:
# ── 5. 保存模型 ────────────────────────────────────────────
os.makedirs('../models', exist_ok=True)
model_path = f'../models/{best_name.lower().replace(" ", "_")}.joblib'
joblib.dump(best_model, model_path)
print(f"[PASS] 模型已保存: {model_path}")

[PASS] 模型已保存: ../models/randomforest.joblib


In [9]:
# ── 6. 写入 metrics.txt ────────────────────────────────────
os.makedirs('../results', exist_ok=True)
metrics_path = '../results/metrics.txt'

with open(metrics_path, 'w',encoding='utf-8') as f:
    f.write("── CV (5-Fold Accuracy)──\n")
    for name, scores in cv_results.items():
        f.write(f"{name:22s}  mean={scores.mean():.4f}  std={scores.std():.4f}\n")
    f.write(f"\n最佳模型: {best_name}\n")
    f.write(f"Hold-out Accuracy: {acc:.4f}\n\n")
    f.write("Classification Report:\n")
    f.write(report)

print(f"[PASS] 指标已写入: {metrics_path}")

[PASS] 指标已写入: ../results/metrics.txt


In [10]:
# ── 7. 断言测试 ────────────────────────────────────────────
print("\n── 断言测试 ──")

assert acc > 0.75, \
    f"[FAIL] Accuracy 过低: {acc:.4f}"
print(f"[PASS] Accuracy 达标: {acc:.4f} > 0.75")

assert os.path.exists(model_path), \
    f"[FAIL] 模型文件不存在: {model_path}"
print(f"[PASS] 模型文件存在: {model_path}")

assert os.path.exists(metrics_path), \
    f"[FAIL] metrics.txt 不存在"
print(f"[PASS] metrics.txt 存在: {metrics_path}")

assert X_test.shape[1] == X_train.shape[1], \
    "[FAIL] 测试集与训练集特征数不一致"
print(f"[PASS] 特征数一致: {X_test.shape[1]} 列")


── 断言测试 ──
[PASS] Accuracy 达标: 0.8156 > 0.75
[PASS] 模型文件存在: ../models/randomforest.joblib
[PASS] metrics.txt 存在: ../results/metrics.txt
[PASS] 特征数一致: 8 列
